In [ ]:
# Check every step of the Recipe Profiling pipeline (tools)

## Check Chromadb Databases

In [ ]:
import chromadb

PERSIST_PATH = "chroma_db"

client = chromadb.PersistentClient(path=PERSIST_PATH)

collections = client.list_collections()

print("Collections in", PERSIST_PATH)
for col in collections:
    print(f"- {col.name} (count={col.count()})")

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("nutritional_ingredients_irish")

sample = collection.peek(limit=3)  # or collection.get(limit=3, include=["metadatas"])
for meta in sample["metadatas"]:
    print(meta.keys())
    print(meta)


## Check Query Tools

In [ ]:
from utils.query_chromadb import get_ingredient_embedding

get_ingredient_embedding("low-fat gouda")

In [ ]:
from utils.query_chromadb import query_sustainability_db

result = query_sustainability_db("low-fat gouda")
print(result)

In [ ]:
from utils.query_chromadb import query_nutritional_db_irish

result = query_nutritional_db_irish("tomatoe")

print(result)

## Parse Recipe Tool

In [ ]:
from tools.parse_recipe_tool import parse_recipe_tool_open

raw_text = """
Garlic Butter Shrimp

Ingredients (2 servings):

200g shrimp (peeled & deveined)

2 tbsp butter

3 garlic cloves, minced

Juice of half a lemon

Salt & pepper to taste

Fresh parsley (optional)

Instructions:

Heat butter in a skillet over medium heat.

Add garlic, cook 30 seconds until fragrant.

Toss in shrimp, season with salt & pepper.

Cook 2–3 minutes per side until pink.

Squeeze lemon juice on top and sprinkle parsley.

Serve with rice, pasta, or crusty bread.
"""

recipe_dict = parse_recipe_tool_open.invoke(raw_text)

print(recipe_dict)

## Check Weight Ingredient Tool

In [ ]:
# Weight Calculator Tool

from tools.ingredient_weight_tool import ingredient_weight_tool_open

ingredient_names = recipe_dict['ingredient_names']
measurements = recipe_dict['measurements']

weights = ingredient_weight_tool_open.invoke({
    "ingredient_names": ingredient_names,
    "measurements": measurements,
})

print(weights)


## Check Embeddings Tool

In [ ]:
# Embeddings tool

from tools.ingredient_embeddings_tool import ensure_ingredients_in_collection
payload = ensure_ingredients_in_collection.invoke({
    "ingredient_names": ["garlic", "olive oil", "pasta", "low-fat gouda", "canned cherries"],
    "state": {"persist_path": "chroma_db", "collection_name": "ingredients", "debug": True}
})

print(payload)

# Make persist path etc in the state in other tools as well
# payload is a dictionary with all that info. think about the output later

In [ ]:
# Check the new added ingredient in the ingredient collection

from utils.query_chromadb import query_sustainability_db, query_nutritional_db_irish, query_ingredients_db

query_ingredients_db("canned cherries")

## Check Sustainability Tool

In [ ]:
from tools.sustainability_calculator import sustainability_tool_chroma

In [ ]:
# --- Run a real test call via the tool interface (.invoke) ---

result = sustainability_tool_chroma.invoke({
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",      # typo on purpose (should match tomato)
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": [
        "400.0g",      # typo on purpose (should match tomato)
        "30.0g",
        "80.0g",
        "10.0g",
        "320.0g",
        "40.0g",
        "5.0g",
    ],
    "weights": [
        400.0,      # typo on purpose (should match tomato)
        30.0,
        80.0,
        10.0,
        320.0,
        40.0,
        5.0,
    ],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,      # your requested similarity threshold
})


In [ ]:
result

## Check Nutritional Tool

In [ ]:
from tools.nutritional_calculator import nutritional_tool_chroma

In [ ]:
# --- Run a real test call via the tool interface (.invoke) ---

result = nutritional_tool_chroma.invoke({
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",      # typo on purpose (should match tomato)
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": [
        "400.0g",      # typo on purpose (should match tomato)
        "30.0g",
        "80.0g",
        "10.0g",
        "320.0g",
        "40.0g",
        "5.0g",
    ],
    "weights": [
        400.0,      # typo on purpose (should match tomato)
        30.0,
        80.0,
        10.0,
        320.0,
        40.0,
        5.0,
    ],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,      # your requested similarity threshold
})


In [ ]:
result

In [ ]:
from utils.query_chromadb import query_nutritional_db_irish

REQUIRED = ["Protein (g)", "Carbohydrate (g)", "Fat (g)"]

def select_first_with_macros(matches):
    for m in matches or []:
        meta = m.get("metadata") or {}
        # treat "stub" rows (only name/type) as unusable
        if set(meta.keys()) <= {"name", "type"}:
            continue
        if all(k in meta for k in REQUIRED):
            return m
    return None

ingredients = ["tomatoe", "olive oil", "onion", "garlic", "spaghetti", "parmesan", "basil"]

for ing in ingredients:
    matches = query_nutritional_db_irish(ing) or []
    picked = select_first_with_macros(matches)

    print(f"\n🔎 Query: {ing}")
    if not matches:
        print("  ❌ No matches")
        continue

    # quick glance at top-3 to see what's coming back
    for i, m in enumerate(matches[:3]):
        mk = list((m.get("metadata") or {}).keys())
        print(f"  cand#{i+1}: doc={m.get('document')!r}, distance={m.get('distance')}, keys={len(mk)}")

    if not picked:
        print("  ⚠️ No candidate with required macros found.")
        continue

    meta = picked.get("metadata") or {}
    print(f"\n  ✅ Using: {picked.get('document')!r} (distance={picked.get('distance')})")
    print("  --- FULL METADATA ---")
    for k, v in meta.items():
        print(f"    {k}: {v}")


## Check Profiling Tool

In [ ]:
from tools.recipe_profiling_tool import Recipe_Profiling_Tool

In [ ]:
payload = {
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": ["400.0g", "30.0g", "80.0g", "10.0g", "320.0g", "40.0g", "5.0g"],
    "weights": [400.0, 30.0, 80.0, 10.0, 320.0, 40.0, 5.0],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,
}

profile = Recipe_Profiling_Tool(payload)
from pprint import pprint
pprint(profile)

## Check Profiling Chain

In [ ]:
from tools.recipe_profiling_chain import Recipe_Profiling_Chain

In [ ]:
sample_recipe = """
Pasta al Pomodoro
Serves 4

Ingredients:
- 400 g tomatoes (ripe)
- 30 g olive oil
- 1 small carrot, finely chopped
- 80 g onion, finely chopped
- 10 g garlic, minced
- 320 g spaghetti
- 40 g parmesan, grated
- 5 g basil leaves
- Salt to taste

Directions:
1) Sweat onion in olive oil until translucent. Add garlic briefly.
2) Add chopped tomatoes, simmer 15–20 min. Season with salt.
3) Cook spaghetti al dente; toss with sauce.
4) Serve with basil and parmesan.
"""

res = Recipe_Profiling_Chain.invoke({"recipe_text": sample_recipe, "debug": True})
print(res.keys())              
print(res.get("recipe_profile_totals"))

In [ ]:
res

## Check Text2Cypher Tool (Old)

In [ ]:
from tools.Text2Cypher_Neo_tool_open import query_recipes_open

results, cypher = query_recipes_open.invoke({
    "query_text": "Tell me a low-fat recipe with chicken and tomatoe under 1 hour",
    "model": "gpt"
})
results

In [ ]:
print(cypher)

In [ ]:
from tools.Text2Cypher_Neo_tool_open import query_recipes_open

results, cypher = query_recipes_open.invoke({
    "query_text": "Tell me a Low-fat recipe with chicken and tomato under 1 hour",
    "model": "open"
})
results

In [ ]:
print(cypher)

## LangChain T2C tool

In [2]:
from tools.Text2Cypher_langgraph_tool import RecipeSearchApp
import os

app = RecipeSearchApp(
    neo4j_uri=os.environ["NEO4J_URI"]
)

result = app.invoke("Give me a low fat recipe with chicken under 120 minutes")
print(result['results'])
print(result['cypher_statement'])

[{'r.title': 'Chicken Noodle Bake', 'title': 'Chicken Noodle Bake', 'nutri_score': 0.5, 'sustainability_score': 3.307360335195531, 'duration': 50.0}, {'r.title': 'Chicken and Herb Tea Finger Sandwiches', 'title': 'Chicken and Herb Tea Finger Sandwiches', 'nutri_score': 0.5, 'sustainability_score': 2.1201017811704834, 'duration': 20.0}, {'r.title': 'Mennonite Leek Soup', 'title': 'Mennonite Leek Soup', 'nutri_score': 0.25, 'sustainability_score': 1.9462112211221128, 'duration': 75.0}, {'r.title': 'Tuscan Soup: Lightenend', 'title': 'Tuscan Soup: Lightenend', 'nutri_score': 0.25, 'sustainability_score': 3.0386339869281036, 'duration': 80.0}, {'r.title': 'Weight Watchers (Ww) Cheese Soup  2pts', 'title': 'Weight Watchers (Ww) Cheese Soup  2pts', 'nutri_score': 0.5, 'sustainability_score': 2.5177196652719664, 'duration': 25.0}, {'r.title': 'Cantaloupe-Chicken Skewers', 'title': 'Cantaloupe-Chicken Skewers', 'nutri_score': 0.5, 'sustainability_score': 0.80275, 'duration': 21.0}, {'r.title':

## Fetch Recipe Info

In [ ]:
from tools.fetch_recipe_info import fetch_recipe_info

info = fetch_recipe_info("Cajun Chicken Salad")

print(info)

# Test chain components

In [ ]:
from tools.Text2Cypher_langgraph_tool import RecipeGraphApp

In [ ]:
from tools.Text2Cypher_langgraph_tool import RecipeGraphApp
import os

app = RecipeGraphApp(
    neo4j_uri=os.environ["NEO4J_URI"]
)
app.run_guardrails("whats the weather today?")

In [ ]:
q = "chicken and rice under 120 minutes"

cypher = app.run_generate_cypher(q)["cypher_statement"]

In [ ]:
cypher

In [ ]:
q = "chicken and rice under 30 minutes"
bad = "MATCH (r:Recip)-[:HAS_INGREDIEN]->(i1:Ingredient), (r)-[:HAS_INGREDIENT]->(i2:Ingredient) WHERE toLower(i1.name) CONTAINS 'chicken' AND toLower(i2.name) CONTAINS 'rice' AND r.Duration < 30 RETURN DISTINCT r.title;"
out = app.run_validate_cypher(q, bad)
out["next_action"] # expect 'correct_cypher'
out["cypher_errors"] # see why

In [ ]:
v1 = app.run_validate_cypher(q, bad)
print(v1["next_action"])
#print(v1["cypher_errors"]) # expect 'correct_cypher' + explicit errors

In [ ]:
c1 = app.run_correct_cypher(q, bad, v1["cypher_errors"])
fixed = c1["cypher_statement"]
print(fixed)

In [ ]:
fixed2 = """MATCH (r:Recipe)-[:HAS_INGREDIENT]->(i1:Ingredient),
(r:Recipe)-[:HAS_INGREDIENT]->(i2:Ingredient)
WHERE toLower(i1.name) CONTAINS 'chicken'
AND toLower(i2.name) CONTAINS 'rice'
AND r.duration < 30
RETURN DISTINCT r.title;
"""

In [ ]:
fixed

In [ ]:
fixed2

In [ ]:
v2 = app.run_validate_cypher(q, fixed)
print(v2["next_action"])
print(v2["cypher_errors"]) # expect 'execute_cypher' and no errors

In [ ]:
exec_out = app.run_execute_cypher(fixed)

In [ ]:
exec_out

In [ ]:
final = app.run_generate_final_answer(q, exec_out["database_records"], fixed)
#print(final["answer"]) # summarized reply
print(final["cypher_statement"])
print(final["steps"])

In [ ]:
from tools.Text2Cypher_langgraph_tool import RecipeGraphApp
import os
app = RecipeGraphApp(neo4j_uri=os.environ["NEO4J_URI"])
q = "give me a greek salad"
out = app.invoke(q)
#app.save_graph_png("recipe_langgraph.png")

out

# Plot Hummus vs Irish

In [ ]:
import pandas as pd

df_f = pd.read_pickle('data/pp_recipes_preprocessed_v3_weighted_and_tagged_1000_profiled.pkl')

In [ ]:
df_f.columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

macro_specs = [
    ("total_protein_g_per_serving_hummus", "total_protein_g_per_serving_irish", "Protein (g)"),
    ("total_carbohydrate_g_per_serving_hummus", "total_carbohydrate_g_per_serving_irish", "Carbs (g)"),
    ("total_fat_g_per_serving_hummus", "total_fat_g_per_serving_irish", "Fat (g)"),
    ("total_energy_kcal_per_serving_hummus", "total_energy_kcal_per_serving_irish", "Energy (kcal)"),
]

def macro_scatter(sub_df):
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()

    for (h_col, i_col, label), ax in zip(macro_specs, axes):
        sns.scatterplot(
            data=sub_df,
            x=h_col,
            y=i_col,
            alpha=0.35,
            s=30,
            linewidth=0,
            edgecolor=None,
            ax=ax,
        )
        min_v = min(sub_df[h_col].min(), sub_df[i_col].min())
        max_v = max(sub_df[h_col].max(), sub_df[i_col].max())
        ax.plot([min_v, max_v], [min_v, max_v], linestyle="--", color="gray", linewidth=1)
        ax.set_title(label)
        ax.set_xlabel("Hummus per serving")
        ax.set_ylabel("Irish per serving")

    plt.tight_layout()
    plt.show()

macro_scatter(df_f)
